# 🧪 Unidad 1: De notebook a sistema

En este notebook se mostrará cómo un flujo de trabajo típico de ciencia de datos puede transformarse progresivamente en una estructura más cercana a un sistema MLOps. La idea no es solo entrenar un modelo, sino comprender por qué un notebook aislado resulta insuficiente cuando se requiere reproducibilidad, trazabilidad y colaboración.


## Objetivos de aprendizaje

Al finalizar este notebook, el estudiante podrá:

- identificar las limitaciones de un notebook experimental;
- reconocer los componentes mínimos de una estructura de proyecto reproducible;
- construir un flujo básico de modelado con datos simulados de tiempo de permanencia;
- diferenciar entre un prototipo exploratorio y un sistema estructurado.


## 1. El punto de partida: un notebook que funciona, pero no escala

En muchos proyectos de Machine Learning, el trabajo inicia en un notebook. Allí se cargan datos, se transforman variables, se entrena un modelo y se revisan métricas. Este enfoque es útil para explorar, pero suele presentar problemas importantes:

- las rutas de archivos quedan escritas manualmente;
- no hay separación clara entre datos, código y artefactos;
- los parámetros del modelo quedan dispersos en distintas celdas;
- el flujo depende del orden en que se ejecutan las celdas.

En esta primera parte construiremos justamente ese flujo experimental para luego transformarlo.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE: int = 42
rng = np.random.default_rng(RANDOM_STATE)


## 2. Construcción de un dataset simulado

Trabajaremos con un conjunto de datos *mock* sobre tiempo de permanencia en una plataforma digital. Este dataset no busca representar toda la complejidad del negocio, pero sí capturar una lógica verosímil para fines didácticos.


In [2]:
n_samples: int = 15000

sites = np.array(["MLA", "MLB", "MLC", "MLM"])
device_os = np.array(["android", "ios", "web"])
segments = np.array(["new", "active", "at_risk", "churned"])
entry_points = np.array(["home", "search", "recommendation", "push", "email"])

data = pd.DataFrame({
    "user_id": np.arange(1, n_samples + 1),
    "site": rng.choice(sites, size=n_samples, p=[0.35, 0.20, 0.15, 0.30]),
    "device_os": rng.choice(device_os, size=n_samples, p=[0.55, 0.25, 0.20]),
    "segment": rng.choice(segments, size=n_samples, p=[0.20, 0.45, 0.20, 0.15]),
    "entry_point": rng.choice(entry_points, size=n_samples, p=[0.25, 0.25, 0.20, 0.20, 0.10]),
    "hour_of_day": rng.integers(0, 24, size=n_samples),
    "day_of_week": rng.integers(0, 7, size=n_samples),
    "historical_avg_session_minutes": rng.uniform(2.0, 35.0, size=n_samples),
    "historical_sessions_last_7d": rng.integers(1, 25, size=n_samples),
    "days_since_last_session": rng.integers(0, 30, size=n_samples),
    "push_received_last_24h": rng.integers(0, 6, size=n_samples),
})

segment_effect = data["segment"].map({
    "new": -1.5,
    "active": 3.5,
    "at_risk": -2.0,
    "churned": -4.0,
})

entry_effect = data["entry_point"].map({
    "home": 1.2,
    "search": 2.0,
    "recommendation": 3.0,
    "push": 1.5,
    "email": 0.8,
})

device_effect = data["device_os"].map({
    "android": 0.5,
    "ios": 0.8,
    "web": 1.5,
})

hour_effect = np.where(data["hour_of_day"].between(18, 22), 2.2, 0.0)
weekend_effect = np.where(data["day_of_week"].isin([5, 6]), 1.3, 0.0)

noise = rng.normal(loc=0.0, scale=2.5, size=n_samples)

data["session_minutes"] = (
    1.5
    + 0.55 * data["historical_avg_session_minutes"]
    + 0.30 * data["historical_sessions_last_7d"]
    - 0.25 * data["days_since_last_session"]
    - 0.40 * data["push_received_last_24h"]
    + segment_effect
    + entry_effect
    + device_effect
    + hour_effect
    + weekend_effect
    + noise
)

data["session_minutes"] = data["session_minutes"].clip(lower=0.5)
data.head()


,user_id,site,device_os,segment,entry_point,hour_of_day,day_of_week,historical_avg_session_minutes,historical_sessions_last_7d,days_since_last_session,push_received_last_24h,session_minutes
0,1,MLM,android,active,recommendation,0,4,34.669451,1,15,4,19.931132
1,2,MLB,android,active,push,19,0,11.777831,14,6,4,15.854436
2,3,MLM,android,churned,search,2,1,32.079445,3,5,1,15.680154
3,4,MLC,web,new,home,17,3,31.659289,3,9,4,17.850008
4,5,MLA,android,new,home,0,1,30.216766,6,25,2,8.952484


## 3. Flujo experimental mínimo

A continuación se construye un flujo típico: selección de variables, partición de datos, entrenamiento y evaluación. Aunque esto puede producir un resultado razonable, todavía estamos lejos de un flujo MLOps real.


In [3]:
target_column: str = "session_minutes"
feature_columns = [
    "site",
    "device_os",
    "segment",
    "entry_point",
    "hour_of_day",
    "day_of_week",
    "historical_avg_session_minutes",
    "historical_sessions_last_7d",
    "days_since_last_session",
    "push_received_last_24h",
]

X = data[feature_columns].copy()
y = data[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

categorical_features = ["site", "device_os", "segment", "entry_point"]
numeric_features = [
    "hour_of_day",
    "day_of_week",
    "historical_avg_session_minutes",
    "historical_sessions_last_7d",
    "days_since_last_session",
    "push_received_last_24h",
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


MAE:  2.0833
RMSE: 2.5923
R²:   0.8711


C:\Users\legion\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## 4. Interpretación del resultado

En un problema de regresión, las métricas tienen una interpretación distinta a las de clasificación:

- **MAE** indica el error absoluto promedio entre el valor real y la predicción.
- **RMSE** penaliza con mayor fuerza errores grandes.
- **R²** resume qué proporción de la variabilidad de la variable objetivo está siendo explicada por el modelo.

Estas métricas pueden lucir adecuadas en un notebook, pero todavía no resuelven el problema de fondo: el flujo sigue siendo manual, poco trazable y difícil de compartir.


## 5. ¿Por qué este notebook todavía no es suficiente?

Aunque el modelo funciona y se obtuvieron métricas, todavía existen varias limitaciones:

1. **Los parámetros están embebidos en el notebook.** Si otra persona quiere cambiar el modelo, debe editar celdas manualmente.
2. **No hay separación entre lógica, datos y resultados.** Todo está concentrado en un solo archivo.
3. **No existe trazabilidad formal del experimento.** No se registran parámetros, métricas ni artefactos en una herramienta de tracking.
4. **No hay versionamiento explícito del dataset.** Si el conjunto de datos cambia, el resultado también cambiará.
5. **La reproducibilidad depende del orden de ejecución.** Un notebook puede funcionar para quien lo creó, pero no necesariamente para el resto del equipo.


## 6. Del notebook a una estructura de proyecto

La transición hacia un sistema más cercano a MLOps comienza con una estructura organizada. Una forma mínima y razonable para este caso sería la siguiente:

```text
session-duration-mlops/
├── data/
│   ├── raw/
│   └── processed/
├── notebooks/
├── src/
│   ├── data/
│   ├── features/
│   ├── models/
│   └── utils/
├── models/
├── outputs/
├── pyproject.toml
├── dvc.yaml
├── params.yaml
└── README.md
```

Esta organización no es decorativa. Su objetivo es separar responsabilidades y hacer que el proyecto sea más claro para cualquier integrante del equipo.


## 7. Componentes mínimos de un sistema MLOps

A partir de la estructura anterior, un flujo más profesional requeriría integrar al menos los siguientes componentes:

- **Git y GitHub** para versionar código y coordinar colaboración.
- **Poetry** para definir un entorno reproducible.
- **DVC** para versionar datos y formalizar pipelines.
- **MLflow** para registrar métricas, parámetros y artefactos.

En otras palabras, el notebook no desaparece, pero deja de ser el centro del sistema. Pasa a ser una herramienta de exploración, mientras que el proyecto estructurado se convierte en la unidad principal de trabajo.


## 8. Ejercicio guiado

Antes de continuar con la siguiente unidad, responde y desarrolla lo siguiente:

1. Identifica tres decisiones de modelado que en este notebook quedaron escritas de forma rígida.
2. Propón cómo moverías esas decisiones a un archivo de configuración.
3. Describe qué carpeta del proyecto debería contener:
   - el dataset original,
   - el pipeline de entrenamiento,
   - los modelos exportados,
   - los resultados de evaluación.
4. Explica por qué este notebook, aunque útil, todavía no representa un flujo MLOps completo.


## 9. Idea clave de cierre

El paso más importante en esta unidad es conceptual:

> un notebook puede ser un buen punto de partida, pero no debe ser el destino final de un proyecto de Machine Learning.

En la siguiente sección se profundizará en el ciclo de vida del modelo y en cómo cada etapa requiere decisiones técnicas que van más allá del entrenamiento.
